In [ ]:
import pandas as pd

from study.data import cohort
from study.io import ROOT, WORK, load, reset_directory
from study.metrics import probability_metrics
from study.models import calibration_summary
from study.plotting import save_table

In [ ]:
settings = [
    ('carevue', 'carevue', 'original'),
    ('carevue', 'metavision', 'original'),
    ('metavision', 'mimiciv', 'shared'),
    ('metavision', 'eicu', 'shared')
]

In [ ]:
reset_directory(ROOT / 'results/tables')

sets = load(WORK / 'feature_sets.pkl')
source_fits = load(WORK / 'models/source.pkl')

full = pd.read_parquet(WORK / 'predictions/full')
physiology = pd.read_parquet(WORK / 'results/physiology.parquet')
observation = pd.read_parquet(WORK / 'results/observation.parquet')
timing = pd.read_parquet(WORK / 'results/timing.parquet')

In [ ]:
# Eligible MIMIC cohorts and the eICU test population
rows = []
for domain in ('carevue', 'metavision', 'mimiciv', 'eicu'):
    population = cohort(domain)
    group = population.loc[population.partition.eq('test')] if domain == 'eicu' else population
    rows.append(dict(
        domain = domain, patients = len(group), deaths = int(group.in_hospital_mortality.sum()),
        mortality = group.in_hospital_mortality.mean(), age_median = group.age.median(),
        age_q25 = group.age.quantile(0.25), age_q75 = group.age.quantile(0.75),
        female = int(group.sex.eq('F').sum()), hospitals = group.hospital_id.nunique() if domain == 'eicu' else 1,
    ))

save_table(pd.DataFrame(rows), 'cohorts')

In [ ]:
# Feature counts
counts = pd.DataFrame([
    dict(tier = tier, original = len(sets['original'][tier]), shared = len(sets['shared'][tier]))
    for tier in ('R0', 'R1', 'R2', 'R3', 'R4')
])

save_table(counts, 'tiers')

In [ ]:
# Raw external performance of shared feature MetaVision models
rows = []
selected = full.loc[
    full.source.eq('metavision') & full.support.eq('shared') & full.tier.isin(['R1', 'R2', 'R3'])
    & full.domain.isin(['mimiciv', 'eicu'])
]
for (domain, tier), group in selected.groupby(['domain', 'tier'], observed = True):
    group = group.loc[group.partition.eq('test')] if domain == 'eicu' else group
    y, risk = group.in_hospital_mortality.to_numpy(), group.risk.to_numpy()
    
    rows.append(dict(domain = domain, tier = tier, **probability_metrics(y, risk), **calibration_summary(y, risk)))

columns = ['domain', 'tier', 'log_loss', 'AUROC', 'Brier', 'calibration_intercept', 'calibration_slope']

save_table(pd.DataFrame(rows)[columns], 'external_performance')

In [ ]:
# Increments with individual unadjusted 95% CIs
increments = pd.concat([physiology, observation], ignore_index = True)
increments = increments.loc[
    increments.source.eq('metavision') & increments.support.eq('shared')
    & increments.domain.isin(['mimiciv', 'eicu'])
]

columns = ['domain', 'contrast', 'weighting', 'difference', 'lower', 'upper']

save_table(increments[columns], 'external_performance_increments')

In [ ]:
# Selection counts
flows = {}
for domain in ('carevue', 'metavision', 'mimiciv', 'eicu'):
    flow = pd.read_csv(WORK / 'data' / domain / 'cohort_flow.csv')
    flows[domain] = flow.loc[flow.unit.eq('icu_stays')].set_index('step')

In [ ]:
carevue, metavision = flows['carevue'], flows['metavision']

rows = [
    ('ICU stays in recording system', carevue.loc['recording_system', 'remaining'],
     metavision.loc['recording_system', 'remaining']),
    ('MetaVision stays from patients with CareVue records', None,
     metavision.loc['exclude_carevue_patients', 'excluded']),
    ('Additional ICU stays per patient', carevue.loc['first_icu_stay', 'excluded'],
     metavision.loc['first_icu_stay', 'excluded']),
    ('Selected ICU stays', carevue.loc['first_icu_stay', 'remaining'],
     metavision.loc['first_icu_stay', 'remaining']),
    ('Age below 18 years', carevue.loc['age_at_least_18', 'excluded'],
     metavision.loc['age_at_least_18', 'excluded']),
    ('ICU stay shorter than 48 hours', carevue.loc['icu_stay_at_least_48_hours', 'excluded'],
     metavision.loc['icu_stay_at_least_48_hours', 'excluded']),
    ('Final eligible patients', carevue.loc['icu_stay_at_least_48_hours', 'remaining'],
     metavision.loc['icu_stay_at_least_48_hours', 'remaining'])
]

selection = pd.DataFrame(rows, columns = ['stage', 'CareVue', 'MetaVision'])
selection = selection.astype({'CareVue': 'Int64', 'MetaVision': 'Int64'})

save_table(selection, 'cohort_flow_mimiciii')

In [ ]:
mimiciv = flows['mimiciv']

rows = [
    ('Initial ICU stays', mimiciv.iloc[0].before),
    ('Earliest possible calendar year before 2013', mimiciv.loc['calendar_year_at_least_2013', 'excluded']),
    ('Additional ICU stays per patient', mimiciv.loc['first_icu_stay', 'excluded']),
    ('Selected ICU stays', mimiciv.loc['first_icu_stay', 'remaining']),
    ('Age below 18 years', mimiciv.loc['age_at_least_18', 'excluded']),
    ('ICU stay shorter than 48 hours', mimiciv.loc['icu_stay_at_least_48_hours', 'excluded']),
    ('Final eligible patients', mimiciv.loc['icu_stay_at_least_48_hours', 'remaining'])
]

save_table(pd.DataFrame(rows, columns = ['stage', 'count']), 'cohort_flow_mimiciv')

In [ ]:
eicu = flows['eicu']

rows = [
    ('Initial ICU stays', eicu.iloc[0].before),
    ('Stays from other hospital admissions per patient', eicu.loc['one_hospital_admission_per_patient', 'excluded']),
    ('Additional ICU stays within selected admissions', eicu.loc['first_icu_stay_in_selected_admission', 'excluded']),
    ('Selected ICU stays', eicu.loc['first_icu_stay_in_selected_admission', 'remaining']),
    ('Age below 18 years', eicu.loc['age_at_least_18', 'excluded']),
    ('ICU stay shorter than 48 hours', eicu.loc['icu_stay_at_least_48_hours', 'excluded']),
    ('Missing hospital outcome', eicu.loc['recorded_hospital_outcome', 'excluded']),
    ('Hospital discharge before 48 hours', eicu.loc['hospital_discharge_at_least_48_hours', 'excluded']),
    ('Final eligible patients', eicu.loc['hospital_discharge_at_least_48_hours', 'remaining'])
]

save_table(pd.DataFrame(rows, columns = ['stage', 'count']), 'cohort_flow_eicu')

In [ ]:
# External eICU comparisons across learner families
learners = pd.read_parquet(WORK / 'results/learners.parquet')

learners = learners.loc[learners.domain.eq('eicu') & learners.calibration.eq('raw') & learners.weighting.eq('patient')]
baseline = increments.loc[increments.domain.eq('eicu') & increments.weighting.eq('patient')]
learner_rows = pd.concat([baseline.assign(learner = 'xgboost'), learners], ignore_index = True)

save_table(learner_rows[['learner', 'contrast', 'difference', 'lower', 'upper']], 'learner_controls')

In [ ]:
# Joint uncertainty using all 30 source refits
joint = pd.read_parquet(WORK / 'results/joint_uncertainty.parquet')

joint = joint.loc[joint.source_fits.eq(30)].copy()
baseline = increments.loc[increments.domain.eq('eicu')]
baseline = baseline[['contrast', 'weighting', 'difference']].rename(columns = {'difference': 'full_source_estimate'})
joint = joint.merge(baseline, on = ['contrast', 'weighting'], validate = 'one_to_one')
joint['favours_richer_percent'] = 100 * joint.fraction_below_zero
columns = ['contrast', 'weighting', 'full_source_estimate', 'median', 'lower', 'upper', 'favours_richer_percent']

save_table(joint[columns], 'joint_uncertainty')

In [ ]:
# External eICU timing effects in each representation
selected = timing.loc[timing.source.eq('metavision') & timing.domain.eq('eicu') & timing.support.eq('shared')]

save_table(selected[['tier', 'weighting', 'contrast', 'difference', 'lower', 'upper']], 'timing_decomposition')

In [ ]:
# Paired comparisons of matched-input and complete-input R3 training
retraining = pd.read_parquet(WORK / 'results/local_retraining.parquet')

columns = ['training', 'restriction', 'weighting', 'difference', 'lower', 'upper', 'holm_p']

save_table(retraining[columns], 'matched_retraining')

In [ ]:
# Full and reduced diverse eICU sources evaluated in the complete MIMIC cohorts
reverse = pd.read_parquet(WORK / 'results/reciprocal.parquet')

reverse = reverse.loc[reverse.condition.isin(['full', 'diverse']) & reverse.group.eq('all')]

save_table(reverse[['condition', 'domain', 'contrast', 'difference', 'lower', 'upper']], 'reverse_transfer')